In [1]:
import sys
import os
import numpy as np
# Add the root directory to the path so Jupyter can find the library
sys.path.append(os.path.abspath('..'))

# Libraries
from options import EuropeanOption, SumPricesCV, ArithmeticAsianOption, GeometricAsianOption
from pricers import MonteCarloPricer, QuasiMCPricer, BSMPricer, ControlVariateMC, GeometricAsianPricer
from models import GBM
from risk import AnalyticalBSMGreeks, FiniteDifferenceGreek

# Test 3/04/2026 - European Call ATM

--- 

A tech stock is about to announce earnings. The market is pricing in massive uncertainty, pushing implied volatility sky-high. You are analyzing the Call leg of an ATM straddle.

The Market & Contract Data:
- European Call Option
- Current Asset Price (S0): 100
- Strike Price (K): 100 (Exactly ATM)
- Time to Maturity (T): 0.25 years (3 months)
- Risk-free Rate (r): 3% (0.03)
- Volatility (sigma): 45% (0.45) - High volatility regime

**TASKS**
1. Price the option with BSM, MC and QMC
2. Calculate the Greeks with BSM and Finite Differences
3. Print the results

In [2]:
#1.  ======================================= Market data and Contract structure
S0 = 100 #current price
K = 100 #strike price
ttm = 3/12 # option expires in 3 months
r = 0.03 # risk free rate
sigma = 0.45 #high volatility

# Define contract
contract = EuropeanOption(
    T=ttm, # pass directly time to maturity, then t_time = 0
    K=K,
    option_type="call"
)


#2.  ======================================= Define pricers and How to model the underlying asset
n_paths = 2**15
n_steps = 1 # we only care about the final price in this situation, there is no point to simulate multiple steps
SEED = 42
alpha=0.95

#underlying asset model
model = GBM(
    S0=S0,
    r=r,
    sigma=sigma,
    t_time=0.0
)

#crude Monte Carlo pricer
mc_pricer = MonteCarloPricer(
    n_paths=n_paths,
    n_steps=n_steps,
    seed=SEED,
    alpha=alpha,
    antithetic=False
)

#Monte Carlo with Antithetic Sampling (Variance Reduction method)
mc_anti_pricer = MonteCarloPricer(
    n_paths=n_paths,
    n_steps=n_steps,
    seed=SEED,
    alpha=alpha,
    antithetic=True
)

#Quasi-Monte Carlo pricer
qmc_pricer = QuasiMCPricer(
    n_paths=n_paths,
    n_steps=n_steps,
    seed=SEED
)

#Black-Scholes-Merton pricer (analytical solutions)
bsm_pricer = BSMPricer()


#3.  ======================================= Compute Option Price and CI
mc_price = mc_pricer.price(
    option=contract,
    model=model
)
ci_mc = list(mc_price.metrics.values())[0]

mc_anti_price = mc_anti_pricer.price(
    option=contract,
    model=model
)
ci_anti_mc = list(mc_anti_price.metrics.values())[0]

qmc_price = qmc_pricer.price(
    option=contract,
    model=model
)

bsm_price = bsm_pricer.price(
    option=contract,
    model=model
)


#4.  ======================================= Compute Greeks
bsm_risk = AnalyticalBSMGreeks()
fd_risk = FiniteDifferenceGreek(
    pricer=mc_anti_pricer, 
    dS_percentage=0.01,
    dr = 0.01,
    dttm = 1/365
) #use Monte Carlo with antithetic sampling as pricer

greeks_analytical = bsm_risk.calculate(
    option=contract,
    model=model
) 

greeks_numerical = fd_risk.greeks_calculator(
    option=contract,
    model=model
)

#5. ======================================= Print Results

title = "QUANT LIBRARY TEST - EUROPEAN CALL ATM (HIGH VOL REGIME)"
subtitle_price = "--- 1. PRICING BENCHAMRK ---"
subtitle_greeks = "--- 2. HEDGING BENCHMARK (GREEKS) ---"

#title
print("="*100)
print(f"{title:^100}")
print("="*100)

print("")

#prices section
print(f"{subtitle_price:^100}")
print("")
print(f"{'Analytical (BSM) Price':<50} : {bsm_price.price:.4f}")
print(f"{'Crude MC Price':<50} : {mc_price.price:.4f} | {alpha*100}% CI : [{ci_mc[0]:.4f}, {ci_mc[1]:.4f}]")
print(f"{'Crude MC (with Antithetic Sampling) Price':<50} : {mc_anti_price.price:.4f} | {alpha*100}% CI : [{ci_anti_mc[0]:.4f}, {ci_anti_mc[1]:.4f}]")
print(f"{'Quasi MC Price':<50} : {qmc_price.price:.4f}")

print("")

#greeks section
print(f"{subtitle_greeks:^100}")
print("")
print(f"{'Analytical (BSM)':>25}  {'Finite Differences (MC with Antithetic Sampling & CRN)':>72}")
print(f"")
print(f"{'Delta':<7} : {greeks_analytical['Delta']:.4f} {greeks_numerical['Delta']:>45.4f}")
print(f"{'Gamma':<7} : {greeks_analytical['Gamma']:.4f} {greeks_numerical['Gamma']:>45.4f}")
print(f"{'Vega':<7} : {greeks_analytical['Vega']:.4f} {greeks_numerical['Vega']:>45.4f}")
print(f"{'Rho':<7} : {greeks_analytical['Rho']:.4f} {greeks_numerical['Rho']:>45.4f}")
print(f"{'Theta':<7} : {greeks_analytical['Theta']:.4f} {greeks_numerical['Theta']:>45.4f}")
print("="*100)


                      QUANT LIBRARY TEST - EUROPEAN CALL ATM (HIGH VOL REGIME)                      

                                    --- 1. PRICING BENCHAMRK ---                                    

Analytical (BSM) Price                             : 9.3024
Crude MC Price                                     : 9.4559 | 95.0% CI : [9.2884, 9.6235]
Crude MC (with Antithetic Sampling) Price          : 9.3139 | 95.0% CI : [9.1816, 9.4462]
Quasi MC Price                                     : 9.3021

                               --- 2. HEDGING BENCHMARK (GREEKS) ---                                

         Analytical (BSM)                    Finite Differences (MC with Antithetic Sampling & CRN)

Delta   : 0.5580                                        0.5567
Gamma   : 0.0175                                        0.0178
Vega    : 19.7361                                       19.8004
Rho     : 11.6237                                       11.5897
Theta   : -19.1574                    

# Test 08/04/2026 - Asian Call Slightly OTM

--- 

**REAL WORLD SCENARIO: The Jet Fuel Hedge**

---

You are sitting on the quantitative commodities desk. A major European airline is terrified of summer travel disruptions causing oil prices to spike. They want to buy a 6-month hedge against rising jet fuel costs.

Because oil prices are highly susceptible to sudden, temporary spikes (or even market manipulation right before an expiry date), the client specifically requests an Arithmetic Asian Call Option. By averaging the daily prices over the next 6 months, they are protected against the overall trend without getting crushed by a single anomalous day.

The energy market is highly volatile right now. Your Head of Trading wants to price this option, but they need an extremely tight confidence interval to ensure the desk doesn't misprice the spread.

**Market Parameters**
- Underlying ($S_0$): $85.00 / barrel
- Strike ($K$): $88.00 (Slightly Out-of-the-Money to make the payoff variance highly sensitive)
- Time to Maturity ($T$): 0.5 years (6 months)
- Risk-Free Rate ($r$): 0.04 (4%)
- Volatility ($\sigma$): 0.42 (42% - High vol environment)
- Observation Frequency ($N$): 126 days (Assuming roughly 21 trading days per month)

**Simulation Constraints**: To prove the power of your architecture, you must run this simulation with a deliberately constrained engine:

- Paths: n_paths = 5000
- Seed: 42 (For exact reproducibility)
- Alpha: 0.95 (95% Confidence Interval)
- Antithetic: False

**DELIVERABLES**
1. The Baseline: What is the Crude MC Arithmetic Asian price and the exact width of its 95% Confidence Interval?
2. The Anchor: What is the exact Analytical Geometric Asian price?
3. The Variance Reduction: What is the Control Variate MC Arithmetic price, the optimal $c^*$, and the new width of its 95% Confidence Interval?
4. The Quant Engineering Bonus (Critical Thinking): Look at the ratio between your Crude CI width and your Control Variate CI width. We know that standard error shrinks at a rate of $\mathcal{O}(1/\sqrt{N})$.Mathematically calculate exactly how many paths the Crude Monte Carlo engine would have needed to run to achieve the exact same tight Confidence Interval you just got with your CV engine. Your M5 MacBook Pro's 10-core CPU is exceptionally fast at matrix operations, but I want you to realize exactly how much brute-force computational overhead your mathematics just bypassed.


In [3]:
# 1. ==================== INPUT DATA & SIMULATION CONSTRAINTS

S0 = 85.0 # current asset price
K = 88.0 # strike price
T = 0.5 # years, i.e. 6 months
t_time = 0 # -> T = time to maturity
r = 0.04 #risk free rate
sigma = 0.42 # volatility
N = 21*6 # consider daily prices for the average --> 21 trading days/month * 6 months
option_type="call" #call option

n_paths = 5_000 # number of trajectories
SEED = 42 # reproducibility
alpha = 0.95 # confidence interval level
antithetic = False # do not overlap different variance reduction strategies

# 2. ==================== DEFINE CONTRACTS & HOW TO MODEL UNDERLYING ASSET PRICE
#Arithmetic Asian Option to price
option = ArithmeticAsianOption(
    T=T,
    K=K,
    N=N,
    option_type=option_type
)

# Analogue option, but considering Geometric mean rather than arithmetic
option_geom = GeometricAsianOption(
    T=T,
    K=K,
    N=N,
    option_type=option_type
)

#How to model the underlying asset 
model = GBM(
    S0=S0,
    r=r,
    sigma=sigma,
    t_time=0.0
)

# 3. ==================== DEFINE PRICERS
# Plain MC
crude_mc_pricer = MonteCarloPricer(
    n_paths=n_paths,
    n_steps=N, 
    seed=SEED,
    alpha=alpha,
    antithetic=antithetic
)
# Exact solution with Geometric Averge
geometric_pricer = GeometricAsianPricer()

# Control Variate pricer
cv_mc_pricer = ControlVariateMC(
    mc_pricer=crude_mc_pricer 
)

# 4. ==================== COMPUTE PAYOFFS

# - Crude Monte Carlo
crude_mc_result = crude_mc_pricer.price(
    option=option,
    model=model
)
crude_mc_price = crude_mc_result.price #payoff according to Plain Monte Carlo, no variance reduction applied
crude_mc_ci = crude_mc_result.metrics["confidence interval"] #confidence intervals associated to Plain Monte Carlo
crude_mc_ciw = crude_mc_ci[1] - crude_mc_ci[0] #confidence interval width associated to Plain Monte Carlo

# - Geometric Option ANALYTICAL Solution
geometric_exact_price = geometric_pricer.price(
    option=option_geom,
    model=model
).price

# - Monte Carlo with Geometric Payoff as Control Variate
cv_geo_mc_results = cv_mc_pricer.price(
    option=option, 
    control_var=option_geom,
    exact_cv_price=geometric_exact_price,
    model=model,
    n_pilot=1_000 
)
cv_geo_mc_price = cv_geo_mc_results.price 
cv_geo_mc_ci = cv_geo_mc_results.metrics["confidence_interval"]
cv_geo_mc_ciw = cv_geo_mc_ci[1] - cv_geo_mc_ci[0] #confidence interval width associated to MC with VR (Sum as control variate)
cv_geo_mc_cstar = cv_geo_mc_results.metrics["c_star"]


# 5. ==================== PRINT RESULTS

title = "QUANT LIBRARY TEST - ASIAN CALL OTM "
subtitle_contract = "--- CONTRACT CHARACTERISTICS ---"
subtitle_sim = "--- SIMULATION PARAMETERS ---"
subtitle_price = "--- PRICING ---"
subtitle_variance = "--- VARIANCE REDUCTION GAIN ---"

#title
print("="*100)
print(f"{title:^100}")
print("="*100)
print("")

#contract section
print(f"{subtitle_contract:^100}")
print("")
print(f"{'Current Asset Price':<25}: {S0}")
print(f"{'Strike Price':<25}: {K}")
print(f"{'Time-to-Maturity':<25}: {T}")
print(f"{'Risk-free-rate':<25}: {r}")
print(f"{'Volatility':<25}: {sigma}")
print(f"{'Observation Frequency':<25}: {N}, daily prices")
print("")

#simulation section
print(f"{subtitle_sim:^100}")
print("")
print(f"{'Number of simulations':<25}: {n_paths}")
print(f"{'Confidence Intervals level':<25}: {alpha*100}%")
print("")

print("="*100)
#prices section
print(f"{subtitle_price:^100}")
print("")
print(f"{'Crude MC Price':<50} : {crude_mc_price:.4f} | {alpha*100}% CI : [{crude_mc_ci[0]:.4f}, {crude_mc_ci[1]:.4f}], width = {crude_mc_ciw:.4f}")
print(f"{'Exact (Geometric Avg) Price':<50} : {geometric_exact_price:.4f}")
print(f"{'Control Variate MC Price':<50} : {cv_geo_mc_price:.4f} | {alpha*100}% CI : [{cv_geo_mc_ci[0]:.4f}, {cv_geo_mc_ci[1]:.4f}], width = {cv_geo_mc_ciw:.4f}, c* = {cv_geo_mc_cstar:.4f}")
print("")

#variance reduction section
print(f"{subtitle_variance:^100}")
print(f"{'Width of Confidence Interval reduced by':<50} : {(1 - (cv_geo_mc_ciw / crude_mc_ciw)) * 100:.4f} %")


                                QUANT LIBRARY TEST - ASIAN CALL OTM                                 

                                  --- CONTRACT CHARACTERISTICS ---                                  

Current Asset Price      : 85.0
Strike Price             : 88.0
Time-to-Maturity         : 0.5
Risk-free-rate           : 0.04
Volatility               : 0.42
Observation Frequency    : 126, daily prices

                                   --- SIMULATION PARAMETERS ---                                    

Number of simulations    : 5000
Confidence Intervals level: 95.0%

                                          --- PRICING ---                                           

Crude MC Price                                     : 4.7275 | 95.0% CI : [4.4818, 4.9732], width = 0.4914
Exact (Geometric Avg) Price                        : 4.5781
Control Variate MC Price                           : 4.8862 | 95.0% CI : [4.8758, 4.8965], width = 0.0207, c* = 1.0596

                                  